### Install Libraries

In [ ]:
# !pip install torch transformers datasets peft trl accelerate bitsandbytes

### Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl.experimental.orpo import ORPOConfig, ORPOTrainer
import torch


In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

### Load Dataset

In [ ]:
# dataset = load_dataset("trl-internal-testing/hh-rlhf-helpful-base-trl-style", split="train")
train_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

### Setting peft and Qlora to fit on GPU

In [ ]:
# QLoRA: 4-bit quantization — fits on a single consumer GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Trainable params: ~20M / 7B  (≈ 0.28%)


### Setting ORPO config and trainer 

In [ ]:
training_args = ORPOConfig(
    output_dir="./orpo-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=8e-5,        # Higher LR fine for LoRA
    beta=0.1,                  # Odds ratio weight λ
    max_length=1024,
    # max_prompt_length=512,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=200,
)


In [ ]:
# No ref_model — cleaner than DPO LoRA setup
trainer = ORPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

### to start training run command below

In [ ]:
trainer.train()

### Saved trained model

In [ ]:
# ============================================================
# SAVE
# ============================================================
model.save_pretrained("./orpo-lora-adapters")
tokenizer.save_pretrained("./orpo-lora-adapters")

### Below Step is to Load trained model and predict based on query

In [ ]:
# ============================================================
# LOAD — Option A: keep adapters (4-bit, low RAM)
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",    # same base ORPO was trained on
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./orpo-lora-adapters")
tokenizer = AutoTokenizer.from_pretrained("./orpo-lora-adapters")

In [ ]:
# ============================================================
# PREDICT
# ============================================================
import torch

def predict(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(predict("Explain gradient descent to a 10-year-old."))